# Agentic ASIC Verification — FIFO Pilot

An AI-driven verification flow where **intelligence does the checking** and deterministic
tools only build, run, transcribe, and filter.

```
User -> Main LLM (you, running this notebook)
        -> Orchestrator agent      : model build  ||  document ingestion   (parallel tool calls)
        -> TestGenRouterLLM        : dispatch min(n, 4) generate_test tool calls, judge gen logs
        -> TestRunRouterLLM        : dispatch min(n, 4) run_test tool calls, judge run logs
        -> CheckerLLM (x4)         : retrieve rules (RAG) + read test log -> PASS / FAIL + reason
        -> RAGAS                   : score retrieval + faithfulness of every verdict
        -> Reporter agent          : final report (waits for RAGAS)
```

Design commitments carried into this build:

- **Two vector collections.** The *test plan* feeds test generation. The *rule book* feeds
  checking. The checker cannot see test-plan expectations; it derives them from rules.
- **Rule retrieval lives inside the checker agent** (its only tool). RAGAS therefore scores
  the checker's real retrieval behavior.
- **Test gen and test run are tool calls** made by router LLMs; each returns compact JSON.
- **One test = one simulation run = one log file.** The file is the identity; no markers.
- **Testbench observes, never judges.** The event monitor produces the entire log.
- **Failure policy.** Build/ingest failure aborts and reports to the user with the log.
  Gen/run failures are non-blocking: reported, excluded downstream.

Hardware kit expected next to this notebook: `hw/` (DUT, harness, monitor, sanity stub),
`docs/` (test plan + rule book PDFs). Both were validated under Verilator 5.x.


In [10]:
from __future__ import annotations

import asyncio
import json
from dataclasses import dataclass, replace
from typing import Iterable, Sequence
import os
import re
import shutil
import subprocess
import textwrap
from datetime import date
from getpass import getpass
from pathlib import Path
from typing import Literal, TypedDict

from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
)
from langchain.messages import ToolMessage
from langchain.tools import tool
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker import HierarchicalChunker
from rank_bm25 import BM25Okapi

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


## Configuration and Budgets

Parallelism and call limits are **application policy**, enforced by middleware —
not prompt pleading. `VER_PARALLEL_LIMIT` is the min(n, 4) dispatch width of both routers.


In [30]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

# Optional LangSmith tracing: per-run latency of every node, agent turn,
# tool call, and judge call at smith.langchain.com. Blank key = tracing off.
_ls_key = os.environ.get("LANGSMITH_API_KEY") \
          or getpass("LangSmith API key (blank = no tracing): ").strip()
if _ls_key:
    os.environ["LANGSMITH_API_KEY"] = _ls_key
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "fifo-agentic-verification")
    print(f"LangSmith tracing: on (project {os.environ['LANGSMITH_PROJECT']})")
else:
    os.environ.pop("LANGSMITH_TRACING", None)
    print("LangSmith tracing: off")

TODAY = date.today().isoformat()

CHAT_MODEL_NAME    = os.environ.get("VER_CHAT_MODEL", "gpt-5.4-mini")
CHECKER_MODEL_NAME = os.environ.get("VER_CHECKER_MODEL", "gpt-5.6-sol")
WRITER_MODEL_NAME  = os.environ.get("VER_WRITER_MODEL", CHAT_MODEL_NAME)
# Model assignment (project owner's): routers/orchestrator/reporter on the
# cheap tier; checker on the flagship; RAGAS judge on a DIFFERENT strong
# model so the judge does not share the checker's blind spots.
JUDGE_MODEL_NAME   = os.environ.get("VER_JUDGE_MODEL", "gpt-5.6-terra")

VER_PARALLEL_LIMIT        = int(os.environ.get("VER_PARALLEL_LIMIT", "4"))
ROUTER_MODEL_CALL_LIMIT   = int(os.environ.get("VER_ROUTER_MODEL_CALL_LIMIT", "8"))
CHECKER_MODEL_CALL_LIMIT  = int(os.environ.get("VER_CHECKER_MODEL_CALL_LIMIT", "6"))
RULE_RETRIEVAL_CALL_LIMIT = int(os.environ.get("VER_RULE_RETRIEVAL_CALL_LIMIT", "3"))

PROJECT_DIR = Path.cwd()
HW_DIR      = PROJECT_DIR / "hw"
DOCS_DIR    = PROJECT_DIR / "docs"
LOG_DIR     = PROJECT_DIR / "logs"
BUILD_DIR   = Path.home() / "fifo_build"
for d in (LOG_DIR, BUILD_DIR):
    d.mkdir(exist_ok=True)

TEST_PLAN_PDF = DOCS_DIR / "snix_fifo_test_plan.pdf"
RULE_BOOK_PDF = DOCS_DIR / "snix_fifo_rule_book.pdf"

# temperature 0: verdicts must be reproducible
llm         = ChatOpenAI(model=CHAT_MODEL_NAME, temperature=0)
checker_llm = ChatOpenAI(model=CHECKER_MODEL_NAME, temperature=0)
writer_llm  = ChatOpenAI(model=WRITER_MODEL_NAME, temperature=0)
judge_llm   = ChatOpenAI(model=JUDGE_MODEL_NAME, temperature=0)
embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"Date: {TODAY}")
print(f"Router/orchestrator model: {CHAT_MODEL_NAME}")
print(f"Checker model: {CHECKER_MODEL_NAME}")
print(f"RAGAS judge model (distinct): {JUDGE_MODEL_NAME}")
print(f"Parallel dispatch width: min(n, {VER_PARALLEL_LIMIT})")
print(f"Hardware kit present: {HW_DIR.exists()} | Docs present: {DOCS_DIR.exists()}")


LangSmith tracing: on (project fifo-agentic-verification)
Date: 2026-07-16
Router/orchestrator model: gpt-5.4-mini
Checker model: gpt-5.6-sol
RAGAS judge model (distinct): gpt-5.6-terra
Parallel dispatch width: min(n, 4)
Hardware kit present: True | Docs present: True


## Typed Handoff Contracts 
these are teh contracts going from one agent to another downstream. 


In [12]:
VerdictValue = Literal["PASS", "FAIL"]
PhaseStatus  = Literal["pass", "fail"]


class BuildReport(BaseModel):  ##did the build work
    ok: bool          # the simulation binary exists after the build attempt
    log_tail: str     # compile-log tail: the evidence the orchestrator judges


class IngestReport(BaseModel):   ##did ingestion work
    ok: bool          # ingestion completed without error


class OrchestratorDecision(BaseModel):   ##the orchestrator's go/no-go after seeing build + ingest, with its reason.
    proceed: bool
    reason: str = Field(description="Judgment over the build and ingest tool results")
    build_log_tail: str = ""


class TestSpec(BaseModel):  ## testplan specifics of the test
    test_id: str            # e.g. T3
    name: str               # e.g. fill_sixteen
    feature: str = ""       # feature under test (F-catalog citation from the plan)
    stimulus: str           # stimulus text extracted from the test plan


class GenResult(BaseModel):  #one TestGenAgent's outcome: which test it built, for which feature, pass/fail, compile-log tail.
    test_name: str = ""     # full name, e.g. T3_fill_sixteen (from retrieval)
    feature: str = ""       # the feature this agent was asked to cover
    status: PhaseStatus
    gen_log_tail: str = ""
    error: str = ""


class GenPhaseReport(BaseModel): ##gen rollup: names ready to run; failures to the reporter.
    passed: list[str] = Field(default_factory=list,
                              description="full test names ready to run")
    failed: list[GenResult] = Field(default_factory=list)


class RunResult(BaseModel):   ##one RunAgent's outcome: which test it ran, pass/fail, log tail.
    test_name: str
    status: Literal["completed", "timeout", "crash", "not_run"]
    log_tail: str = ""
    error: str = ""


class RunPhaseReport(BaseModel):  ##run rollup: names with usable logs; failures to the reporter.
    done: list[str] = Field(default_factory=list,
                            description="full test names with usable logs")
    failed: list[RunResult] = Field(default_factory=list)


class RuleCitation(BaseModel):  ##one checker's outcome: which rule it checked, what the rule required vs what the log shows
    rule_id: str            # e.g. R4
    finding: str            # what the rule required vs what the log shows

class CheckVerdict(BaseModel):  ##one checker's outcome: which test it checked, pass/fail, reason.
    test_name: str
    verdict: VerdictValue
    reason: str = Field(
        default="",
        description="Required when verdict is FAIL: violated rule + log evidence",
    )
    citations: list[RuleCitation] = Field(default_factory=list)

class CheckRecord(BaseModel):  ##one checker's outcome: verdict plus its RAGAS tuple (stored separately).
    """One checker invocation: verdict plus its RAGAS tuple (stored separately)."""
    verdict: CheckVerdict
    retrieval_query: str = ""
    retrieved_rule_ids: list[str] = Field(default_factory=list)
    retrieved_chunks: list[str] = Field(default_factory=list)
    test_log: str = ""
    errors: list[str] = Field(default_factory=list)


class CheckPhaseReport(BaseModel): #check rollup: judged vs couldn't-judge.
    checked: list[str] = Field(default_factory=list,
                               description="test_names whose check completed")
    failed_to_check: list[str] = Field(default_factory=list)


class FinalReport(BaseModel): #the reporter's output: summary, results table, failure detail, trust notes.
    title: str
    summary: str
    results_table_markdown: str
    failures_detail_markdown: str = ""
    trust_notes: str = Field(default="", description="RAGAS-based trust commentary")


class PipelineState(TypedDict, total=False): #the shared state each stage writes its report into.
    user_query: str                     # the features the user wants tested
    orchestrator: OrchestratorDecision
    build: BuildReport
    ingest: IngestReport
    gen_phase: GenPhaseReport
    run_phase: RunPhaseReport
    check_records: list[CheckRecord]
    ragas_summary: dict
    report: FinalReport
    errors: list[str]

print("Contracts defined.")


Contracts defined.


## Deterministic Tools: Build and Simulate

Plain code. `verilator_build` compiles DUT + harness + one test module into a per-test
binary. `run_simulation` executes it and captures the monitor's event stream as the
test's log file — one run, one file, the file is the test's identity.


In [31]:
VERILATOR = shutil.which("verilator") or shutil.which("verilator-cli") or "verilator"  ## shutil.which is a function that returns the path of a command
HW_SOURCES = ["snix_sync_fifo.sv", "fifo_event_monitor.sv", "tb_top.sv"]  ## the source files of the FIFO hardware being tested.
SIM_TIMEOUT_S = 60


def run_cmd(cmd: list[str], cwd: Path, timeout: int = 300) -> tuple[int, str]: #this function runs a command and returns the return code and the output.
    proc = subprocess.run(
        cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout
    )
    return proc.returncode, (proc.stdout + proc.stderr)


def tail(text: str, lines: int = 25) -> str:   ## captures the last n lines of a string.
    return "\n".join(text.strip().splitlines()[-lines:])


def sanitize_name(name: str) -> str:
    """Normalize hand-typed names ("T7 — lap_the_buffer") to pipeline form
    (T7_lap_the_buffer). Bare names ("lap_the_buffer") auto-complete from the
    testbench's test labels when unambiguous."""
    name = re.sub(r"_+", "_", re.sub(r"[^A-Za-z0-9_]+", "_", name.strip())).strip("_")
    if re.match(r"^T\d+_", name):
        return name
    labels = sorted(set(re.findall(r'"(T\d+_\w+)"', (HW_DIR / "tb_top.sv").read_text())))
    hits = [l for l in labels if l == name or l.endswith(f"_{name}") or l.split("_", 1)[-1] == name]
    if len(hits) == 1:
        return hits[0]
    return name  # unknown/ambiguous: leave as typed; downstream reports it


def verilator_build(work_name: str) -> BuildReport:  ## this function compiles the DUT + monitor + the plusarg tb_top (which holds the pre-decided test sequences) into build/<work_name>/.
    """One-line test generation: compile DUT + monitor + the plusarg tb_top
    (which holds the pre-decided test sequences) into build/<work_name>/."""
    work_name = sanitize_name(work_name)
    work = BUILD_DIR / work_name
    if work.exists():
        shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True, exist_ok=True)
    for src in HW_SOURCES:
        shutil.copy(HW_DIR / src, work)

    cmd = [
        VERILATOR, "--binary", "--timing", "-Wall",
        "-Wno-DECLFILENAME", "-Wno-SYNCASYNCNET", "-Wno-BLKSEQ",
        "tb_top.sv", *HW_SOURCES[:2],
        "--top-module", "tb_top",
    ]
    code, log = run_cmd(cmd, cwd=work)
    binary = work / "obj_dir" / "Vtb_top"
    if not binary.exists():
        # some builds need an explicit make pass (observed with pip verilator)
        mk = work / "obj_dir" / "Vtb_top.mk"
        if mk.exists():
            code2, log2 = run_cmd(
                ["make", "-s", "-C", str(work / "obj_dir"),
                 "-f", "Vtb_top.mk", "CXX=c++ -fcoroutines"],
                cwd=work,
            )
            log += "\n" + log2
    ok = binary.exists()
    return BuildReport(ok=ok, log_tail=tail(log))


def run_simulation(test_name: str) -> RunResult:  ## this function runs a simulation of the test.
    test_name = sanitize_name(test_name)
    binary = BUILD_DIR / test_name / "obj_dir" / "Vtb_top"
    if not binary.exists():
        return RunResult(test_name=test_name, status="not_run",
                         error="binary missing; generation failed upstream")
    log_path = LOG_DIR / f"{test_name}.log"
    try:
        proc = subprocess.run(
            [str(binary), f"+TEST={test_name}"],
            capture_output=True, text=True, timeout=SIM_TIMEOUT_S
        )
        text = "\n".join(
            line for line in proc.stdout.splitlines() if not line.startswith("-")
        )
        log_path.write_text(text)
        if "status=completed" in text:
            status = "completed"
        elif "status=timeout" in text:
            status = "timeout"
        else:
            status = "crash"
        return RunResult(test_name=test_name, status=status,
                         log_tail=tail(text, 40))
    except subprocess.TimeoutExpired:
        return RunResult(test_name=test_name, status="timeout",
                         error=f"wall-clock timeout after {SIM_TIMEOUT_S}s")
    except Exception as exc:
        return RunResult(test_name=test_name, status="crash",
                         error=f"{type(exc).__name__}: {exc}")

print(f"Verilator: {VERILATOR}")


Verilator: /usr/bin/verilator


### Gate 1 — model build (no LLM, no retrieval)

Compiles DUT + monitor + `tb_top` once into a scratch work dir. Pass = `ok=True`
with a usable binary. Run this before spending anything on the stages below.


In [32]:
# GATE 1: model build smoke — deterministic, free
gate_build = verilator_build("gate_smoke")
print(f"ok={gate_build.ok}")
print(gate_build.log_tail if not gate_build.ok else "binary built: build/gate_smoke/obj_dir/Vtb_top")


ok=True
binary built: build/gate_smoke/obj_dir/Vtb_top


## Ingestion and Hybrid Retrieval 

Two in-memory Qdrant collections — `fifo_testplan` and `fifo_rules` — built once per
run by the orchestrator's ingest tool. The PDFs are parsed by **Docling** and split
with its `HierarchicalChunker`; chunks are grouped by the T/R id in their heading
breadcrumb, giving one breadcrumb-prefixed chunk per test (9) and per rule (11),
each carrying its `test_id` / `rule_id`.
Retrieval reads the stores back through dense (`similarity_search_with_score`) and
lexical (`BM25Okapi`) sides, fused with reciprocal rank fusion:
**testplan → top 1** (the test to generate), **rules → top 2** (into the checker).


In [ ]:
# --- Docling hierarchical chunking ------------------------------------------
# The PDFs are parsed by Docling's layout model; HierarchicalChunker emits one
# chunk per paragraph tagged with its heading breadcrumb. Chunks are grouped by
# the T/R id in their innermost matching heading: one section per test / rule.

TEST_HEADING = re.compile(r"(T\d+)\s*[—-]\s*(\w+)")
RULE_HEADING = re.compile(r"\b(R\d+)\b")


def load_document(source: Path):
    """Parse a PDF using Docling. Returns a DoclingDocument, not a string."""
    converter = DocumentConverter()
    result = converter.convert(str(source))
    return result.document


def convert_chunk(doc_chunk) -> dict:  # create a dict with headers and text beneath it.
    """
    Convert a Docling DocChunk into a plain dict.

    headings   -> list preserved as-is
    content    -> paragraph text
    chunk_text -> breadcrumb + content  (what gets embedded)
    """
    headings   = doc_chunk.meta.headings or []
    content    = doc_chunk.text.strip()
    breadcrumb = " > ".join(headings)
    chunk_text = f"{breadcrumb}\n\n{content}" if breadcrumb else content
    return {"headings": headings, "content": content, "chunk_text": chunk_text}


def group_sections(chunks: Iterable[dict], pattern: re.Pattern) -> dict[str, dict]: ## this function groups the chunks by the innermost heading matching `pattern`. so if heading is 4.3 T3 and 4 chunks with this header , then you combine all of them as they are part of the same test.
    """Group chunk dicts by the innermost heading matching `pattern`.
    Returns {section_id: {match, breadcrumb, parts}} in document order."""
    sections: dict[str, dict] = {}
    for chunk in chunks:
        match = None
        for heading in chunk["headings"]:
            m = pattern.search(heading)
            if m:
                match = m
        if match is None:
            continue
        section = sections.setdefault(match.group(1), {
            "match": match,
            "breadcrumb": " > ".join(chunk["headings"]),
            "parts": [],
        })
        section["parts"].append(chunk["content"])
    return sections


def docling_sections(path: Path, pattern: re.Pattern) -> dict[str, dict]:  ## this function loads the document, chunks it, and groups the chunks by the innermost heading matching `pattern` and returns a dictionary of header and the chunks.
    document = load_document(path)
    chunks = [convert_chunk(c) for c in HierarchicalChunker().chunk(document)]
    return group_sections(chunks, pattern)


def _section_text(section: dict) -> str:  ## this function returns the breadcrumb and the text beneath it.
    return section["breadcrumb"] + "\n" + "\n".join(section["parts"])


def parse_test_plan(path: Path) -> list[dict]:
    """Chunk labels only — no TestSpec here; ingestion just ingests."""
    out = []
    for tid, section in docling_sections(path, TEST_HEADING).items():
        body = _section_text(section)
        out.append({"test_id": tid, "name": section["match"].group(2),
                    "body": " ".join(body.split())})
    return out


def parse_rule_book(path: Path) -> list[tuple[str, str]]:
    return [(rid, " ".join(_section_text(section).split()))
            for rid, section in docling_sections(path, RULE_HEADING).items()]


# --- retrieval contract and helpers ----------------------------------------

@dataclass(frozen=True)
class RetrievedDocument:
    id: str
    text: str
    score: float | None
    evidence_ids: tuple[str, ...]
    metadata: dict


def as_retrieved_document(document: Document, score: float | None = None) -> RetrievedDocument:
    return RetrievedDocument(
        id=document.metadata["chunk_id"],
        text=document.page_content,
        score=float(score) if score is not None else None,
        evidence_ids=(document.metadata["page_id"],),
        metadata=dict(document.metadata),
    )


def print_results(results: Sequence[RetrievedDocument], text_limit: int = 260) -> None:
    for rank, result in enumerate(results, start=1):
        section = result.metadata.get("page_id", "?")
        score = "n/a" if result.score is None else f"{result.score:.4f}"
        print(f"#{rank} | {result.id} | section={section} | score={score}")
        print(result.text[:text_limit].replace("\n", " "))
        print()


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


FIRST_STAGE_K = 2

TP_CHILDREN: list[Document] = []
RB_CHILDREN: list[Document] = []
TP_STORE = RB_STORE = None
TP_BM25 = RB_BM25 = None


def _section_documents() -> tuple[list[Document], list[Document]]:
    """One Document per test / per rule section; children inherit metadata."""
    tests = parse_test_plan(TEST_PLAN_PDF)
    rules = parse_rule_book(RULE_BOOK_PDF)

    testplan_sections = [
        Document(page_content=t["body"], metadata={
            "source": TEST_PLAN_PDF.name, "document_type": "test_plan",
            "page_id": t["test_id"], "test_id": t["test_id"],
            "name": t["name"],
        })
        for t in tests
    ]
    rule_sections = [
        Document(page_content=body, metadata={
            "source": RULE_BOOK_PDF.name, "document_type": "rule_book",
            "page_id": rule_id, "rule_id": rule_id,
        })
        for rule_id, body in rules
    ]
    return testplan_sections, rule_sections


def build_collections() -> IngestReport:
    global TP_CHILDREN, RB_CHILDREN, TP_STORE, RB_STORE, TP_BM25, RB_BM25
    try:
        testplan_sections, rule_sections = _section_documents()

        TP_CHILDREN = list(testplan_sections)   # one Docling section per chunk
        for index, child in enumerate(TP_CHILDREN):
            child.metadata["chunk_id"] = f"tp-child-{index:03d}"
        TP_STORE = QdrantVectorStore.from_documents(
            documents=TP_CHILDREN, embedding=embeddings,
            location=":memory:", collection_name="fifo_testplan",
        )
        TP_BM25 = BM25Okapi([tokenize(child.page_content) for child in TP_CHILDREN])

        RB_CHILDREN = list(rule_sections)       # one Docling section per chunk
        for index, child in enumerate(RB_CHILDREN):
            child.metadata["chunk_id"] = f"rb-child-{index:03d}"
        RB_STORE = QdrantVectorStore.from_documents(
            documents=RB_CHILDREN, embedding=embeddings,
            location=":memory:", collection_name="fifo_rules",
        )
        RB_BM25 = BM25Okapi([tokenize(child.page_content) for child in RB_CHILDREN])

        if len(testplan_sections) < 9:
            print(f"warning: only {len(testplan_sections)} tests parsed; expected 9")
        if len(rule_sections) < 11:
            print(f"warning: only {len(rule_sections)} rules parsed; expected 11")
        return IngestReport(ok=bool(testplan_sections and rule_sections))
    except Exception as exc:
        print(f"ingest error: {type(exc).__name__}: {exc}")
        return IngestReport(ok=False)


# --- reading the stores back at retrieval time ------------------------------

def dense_retrieve_testplan(question: str, k: int = 2) -> list[RetrievedDocument]:
    matches = TP_STORE.similarity_search_with_score(question, k=k)
    return [as_retrieved_document(document, score) for document, score in matches]


def dense_retrieve_rules(question: str, k: int = 2) -> list[RetrievedDocument]:
    matches = RB_STORE.similarity_search_with_score(question, k=k)
    return [as_retrieved_document(document, score) for document, score in matches]


def bm25_retrieve_testplan(question: str, k: int = 2) -> list[RetrievedDocument]:
    scores = TP_BM25.get_scores(tokenize(question))
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [as_retrieved_document(TP_CHILDREN[i], float(scores[i])) for i in ranked[:k]]


def bm25_retrieve_rules(question: str, k: int = 2) -> list[RetrievedDocument]:
    scores = RB_BM25.get_scores(tokenize(question))
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [as_retrieved_document(RB_CHILDREN[i], float(scores[i])) for i in ranked[:k]]


def reciprocal_rank_fusion(
    ranked_lists: Iterable[Sequence[RetrievedDocument]],
    *,
    limit: int,
    rrf_constant: int = 60,
) -> list[RetrievedDocument]:
    scores: dict[str, float] = {}
    documents_by_id: dict[str, RetrievedDocument] = {}
    for ranked_list in ranked_lists:
        for rank, document in enumerate(ranked_list, start=1):
            documents_by_id.setdefault(document.id, document)
            scores[document.id] = scores.get(document.id, 0.0) + 1 / (rrf_constant + rank)
    return [
        replace(
            documents_by_id[document_id],
            score=score,
            metadata={**documents_by_id[document_id].metadata, "rrf_score": score},
        )
        for document_id, score in
        sorted(scores.items(), key=lambda item: item[1], reverse=True)[:limit]
    ]


def hybrid_testplan_retrieve(question: str, k: int = 1) -> list[RetrievedDocument]:
    return reciprocal_rank_fusion(
        [dense_retrieve_testplan(question, k=FIRST_STAGE_K),
         bm25_retrieve_testplan(question, k=FIRST_STAGE_K)],
        limit=k,
    )


def hybrid_rules_retrieve(question: str, k: int = 2) -> list[RetrievedDocument]:
    return reciprocal_rank_fusion(
        [dense_retrieve_rules(question, k=FIRST_STAGE_K),
         bm25_retrieve_rules(question, k=FIRST_STAGE_K)],
        limit=k,
    )


def hybrid_rule_search(query: str, k: int = 2) -> list[dict]:
    """Query-seeded rules and the retrieve_rules tool share this."""
    return [{"rule_id": d.metadata.get("rule_id", "?"), "text": d.text}
            for d in hybrid_rules_retrieve(query, k=k)]


TESTGEN_RECORDS: dict[str, dict] = {}   # side channel -> RAGAS testgen scoring


@tool(
    "retrieve_test",
    description=(
        "Retrieve the test from the ingested test plan that covers a feature. "
        "Input: the feature to test, in natural language. Returns the matching "
        "test_name and its feature line."
    ),
)
def retrieve_test(feature_query: str) -> str:
    results = hybrid_testplan_retrieve(feature_query, k=1)
    if not results:
        return json.dumps({})
    hit = results[0]
    m = hit.metadata
    # TestSpec exists only from this moment on, built from the chosen chunk
    fm = re.search(r"Feature under test:\s*(.+?)(?:Description:|$)", hit.text, re.DOTALL)
    sm = re.search(r"Stimulus:\s*(.+?)(?:Expected results:|Pass criteria:|$)", hit.text, re.DOTALL)
    spec = TestSpec(test_id=m["test_id"], name=m["name"],
                    feature=" ".join(fm.group(1).split()) if fm else "",
                    stimulus=" ".join(sm.group(1).split()) if sm else "")
    test_name = f"{spec.test_id}_{spec.name}"
    TESTGEN_RECORDS[test_name] = {          # side channel: RAGAS scores this retrieval
        "user_input": feature_query,
        "feature": spec.feature,
        "retrieved_contexts": [hit.text],
        "response": test_name,
    }
    return json.dumps({"test_name": test_name, "feature": spec.feature}, indent=1)


@tool(
    "retrieve_rules",
    description=(
        "Retrieve the FIFO rule-book sections most relevant to a checking "
        "question, using hybrid BM25 + dense search. Input: a natural-language "
        "query about the behavior under judgment. Returns rule chunks tagged "
        "with rule_id."
    ),
)
def retrieve_rules(query: str) -> str:
    return json.dumps(hybrid_rule_search(query), indent=1)

print("Two Qdrant collections + BM25 indexes defined; hybrid retrieval ready.")


Two Qdrant collections + BM25 indexes defined; hybrid retrieval ready.


### Gate 2 — ingestion + retrieval probes

Builds both collections (Docling → Qdrant + BM25) and reports counts:
expect 9 testplan records, 11 rules, empty notes. Then two interactive probes —
type a feature query (top-1 test) and a checking query (top-2 rules); blank skips.


In [16]:
# GATE 2: ingestion — embedding cost only
gate_ingest = build_collections()
print(gate_ingest.model_dump_json(indent=1))

q = input("probe testplan (feature to test, blank=skip): ").strip()
if q:
    print_results(hybrid_testplan_retrieve(q, k=1))
q = input("probe rules (checking question, blank=skip): ").strip()
if q:
    print_results(hybrid_rules_retrieve(q, k=2))


[INFO] 2026-07-16 00:29:43,205 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-16 00:29:43,228 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-16 00:29:43,239 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.1/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-16 00:29:45,376 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-07-16 00:29:48,885 [RapidOCR] download_file.py:95: Successfully saved to: /home/sajalosta/.venvs/fifo/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-16 00:29:48,888 [RapidOCR] main.py:50: Using /home/sajalosta/.venvs/fifo/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-16 00:29:49,174 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-16 00:29:49,176 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-16 00:29:49,177 [RapidOCR] download_file

{
 "ok": true
}


## Orchestrator Agent

Two tools — sanity model build and document ingestion — emitted **in one model turn**
so they execute in parallel. The judgment ("both OK? proceed / build fail: stop and
surface the log") is the LLM's; the tools only report facts. On failure the graph
routes straight to the reporter, which addresses the user (the Main-LLM path).


In [17]:
@tool(
    "build_model",
    description=(
        "Compile the DUT with the testbench (sanity build). Returns a "
        "BuildReport JSON: ok flag and compile-log tail."
    ),
)
def build_model(reason: str = "sanity build") -> str:
    report = verilator_build("sanity")
    return report.model_dump_json(indent=1)


@tool(
    "ingest_documents",
    description=(
        "Parse the test-plan and rule-book PDFs into the two vector "
        "collections. Returns an IngestReport JSON."
    ),
)
def ingest_documents(reason: str = "ingest docs") -> str:
    return build_collections().model_dump_json(indent=1)


ORCHESTRATOR_PROMPT = f"""
You are the orchestrator of an ASIC verification pipeline. Today is {TODAY}.

Call BOTH tools in your first response, in the same turn, so they run in
parallel: build_model and ingest_documents.

Then judge the results:
- Read the build log tail the way a verification engineer reads a compile log.
  A build can be unusable even when ok=true (empty binary, fatal warnings).
- Read the ingest report. Missing tests or rules make checking impossible.

Return OrchestratorDecision. proceed=true only if both results are usable.
If the build failed, set proceed=false and put the build log tail in
build_log_tail with a plain-language reason: the pipeline must stop and the
user must see why. Do not invent log content. Test selection is NOT your
job; the feature request passes through to the test generation stage.
"""

orchestrator_agent = create_agent(
    model=llm,
    tools=[build_model, ingest_documents],
    system_prompt=ORCHESTRATOR_PROMPT,
    middleware=[
        ModelCallLimitMiddleware(run_limit=4, exit_behavior="end"),
        ToolCallLimitMiddleware(tool_name="build_model", run_limit=2,
                                exit_behavior="continue"),
        ToolCallLimitMiddleware(tool_name="ingest_documents", run_limit=2,
                                exit_behavior="continue"),
    ],
    response_format=OrchestratorDecision,
    name="orchestrator",
)
print("Orchestrator agent ready.")


Orchestrator agent ready.


## Test Generation: Router -> TestGenAgents -> Tools

The user's feature request flows to the **TestGenRouterLLM**, which determines how
many tests are needed and calls that many **TestGenAgents** (min(n, 4) per wave).

Each TestGenAgent is an LLM with two tools, called in sequence:
1. `retrieve_test` — hybrid retrieval over the testplan collection: feature -> test_name
2. `gen_test` — one-line Verilator compile of the plusarg testbench for that test_name
   (generate + compile; the compile log is the gen log)

The agent returns its GenResult to the router, which judges each: pass -> run stage,
fail -> reporter. Test names live in the test plan and the testbench — never in cells.


In [18]:
@tool(
    "gen_test",
    description=(
        "Generate one test: a one-line Verilator compile of the testbench for "
        "the given test_name (e.g. T3_fill_sixteen). Returns the compile "
        "report; its log tail is the gen log."
    ),
)
def gen_test(test_name: str) -> str:
    report = verilator_build(test_name.strip())
    return report.model_dump_json(indent=1)


TESTGEN_AGENT_PROMPT = f"""
You are a TestGenAgent in an ASIC verification pipeline. Today is {TODAY}.

You receive ONE feature to test. Work in exactly this order:
1. Call retrieve_test with the feature. It returns the matching test_name
   from the test plan.
2. When the retrieval result arrives, call gen_test with that exact
   test_name.
3. When the build report arrives, return GenResult: the test_name, the
   feature, status pass if the build was ok (usable binary), fail otherwise,
   and the gen log tail. Never invent test names or log content.
"""

testgen_agent = create_agent(
    model=llm,
    tools=[retrieve_test, gen_test],
    system_prompt=TESTGEN_AGENT_PROMPT,
    middleware=[
        ModelCallLimitMiddleware(run_limit=5, exit_behavior="end"),
        ToolCallLimitMiddleware(tool_name="retrieve_test", run_limit=2,
                                exit_behavior="continue"),
        ToolCallLimitMiddleware(tool_name="gen_test", run_limit=2,
                                exit_behavior="continue"),
    ],
    response_format=GenResult,
    name="testgen_agent",
)


@tool(
    "run_testgen_agent",
    description=(
        "Delegate ONE feature to a TestGenAgent, which retrieves the matching "
        "test and generates (compiles) it. Input: the feature to test, in "
        "natural language. Returns the agent's GenResult JSON."
    ),
)
async def run_testgen_agent(feature: str) -> str:
    try:
        result = await testgen_agent.ainvoke(
            {"messages": [{"role": "user",
                           "content": f"Feature to test: {feature}"}]}
        )
        gen = GenResult.model_validate(result["structured_response"])
        if not gen.feature:
            gen = gen.model_copy(update={"feature": feature})
        return gen.model_dump_json(indent=1)
    except Exception as exc:
        return GenResult(feature=feature, status="fail",
                         error=f"{type(exc).__name__}: {exc}"
                         ).model_dump_json(indent=1)


TESTGEN_ROUTER_PROMPT = f"""
You are TestGenRouterLLM in an ASIC verification pipeline. Today is {TODAY}.

You receive the user's request naming the features to test. Determine how
many tests are needed: one TestGenAgent call per distinct feature. A request
for everything ("all features", "full regression") means one call per
feature in the request context.

Work in waves: each model turn, emit min(remaining, {VER_PARALLEL_LIMIT})
run_testgen_agent tool calls TOGETHER so they execute in parallel, one
feature per call, until every feature has been attempted exactly once.

Judge each GenResult like a verification engineer reading a compile log:
status pass with a clean build means the test generated; anything else
failed. Return GenPhaseReport: passed = the full test_names that generated
(they move to the run stage); failed = failed GenResults verbatim (they go
to the reporter). One failure never blocks the rest.
"""

testgen_router = create_agent(
    model=llm,
    tools=[run_testgen_agent],
    system_prompt=TESTGEN_ROUTER_PROMPT,
    middleware=[
        ModelCallLimitMiddleware(run_limit=ROUTER_MODEL_CALL_LIMIT,
                                 exit_behavior="end"),
        ToolCallLimitMiddleware(tool_name="run_testgen_agent", run_limit=12,
                                exit_behavior="continue"),
    ],
    response_format=GenPhaseReport,
    name="testgen_router",
)
print("TestGenAgent + TestGenRouterLLM ready.")


TestGenAgent + TestGenRouterLLM ready.


### Gate 3 — gen_test tool, direct call (no LLM)

Type a test name from your test plan; the tool compiles its binary.
The name is remembered for gates 4–5.


In [19]:
# GATE 3: one gen_test tool call — deterministic, free
GATE_TEST = input("test name to generate (from the test plan, blank=skip): ").strip()
if GATE_TEST:
    print(gen_test.invoke({"test_name": GATE_TEST}))


## Test Run: a Tool Call the Router Makes

`run_test` executes one per-test binary and captures the monitor's event stream into
`logs/<test_id>.log`. The router judges the run log tail (completed / timeout / crash),
streams each done test toward checking, and reports tests that didn't run.


In [20]:
@tool(
    "run_test",
    description=(
        "Run one previously generated test binary. Input: the full "
        "test_name (e.g. T3_fill_sixteen). Returns RunResult JSON with the "
        "log tail to judge."
    ),
)
def run_test(test_name: str) -> str:
    result = run_simulation(test_name.strip())
    return result.model_dump_json(indent=1)


TESTRUN_ROUTER_PROMPT = f"""
You are TestRunRouterLLM in an ASIC verification pipeline. Today is {TODAY}.

You receive the list of test_names that passed generation.
Work in waves: each model turn, emit min(remaining, {VER_PARALLEL_LIMIT})
run_test tool calls TOGETHER for parallel execution, until every test has
been run exactly once.

Pass/fail judgment belongs to the checking phase, never to you. Both
status=completed and status=timeout leave a log, so both stream to checking
(the checkers judge a timeout as a failed test).

Return RunPhaseReport: done = test_names whose run produced a log (completed
or timeout); failed = RunResults where nothing executed (crash, not_run),
verbatim for the reporter. One test failing to run never blocks the others.
"""

testrun_router = create_agent(
    model=llm,
    tools=[run_test],
    system_prompt=TESTRUN_ROUTER_PROMPT,
    middleware=[
        ModelCallLimitMiddleware(run_limit=ROUTER_MODEL_CALL_LIMIT,
                                 exit_behavior="end"),
        ToolCallLimitMiddleware(tool_name="run_test", run_limit=12,
                                exit_behavior="continue"),
    ],
    response_format=RunPhaseReport,
    name="testrun_router",
)
print("Test run router ready.")


Test run router ready.


### Gate 4 — run_test tool, direct call (no LLM)

Runs the binary from gate 3 and writes `logs/<test_name>.log`.
Pass = `status=completed` and `[EVT]` lines in the log tail.


In [21]:
# GATE 4: one run_test tool call — deterministic, free
name = globals().get("GATE_TEST") or input("test name to run (blank=skip): ").strip()
if name:
    GATE_TEST = name
    print(run_test.invoke({"test_name": name}))


## The Checker Agent — One Feature-Driven Retrieval

The heart of the project. Each checker invocation gets **one test's log** plus the
rule-book sections retrieved **once**, using the test's feature (captured at testgen
retrieval) as the query. The checker has no tools: it derives what the provided rules
require from the observed stimulus, judges compliance, and returns a binary verdict
with rule citations.

Guardrails: closed world (judge only from provided rules + log), citation-or-nothing,
temperature 0, missing evidence = FAIL with that stated. A deterministic sanitizer then
enforces provenance: every cited rule_id must appear in the retrieved set.


In [ ]:
CHECKER_PROMPT = f"""
You are a hardware verification checker. Today is {TODAY}.

INPUT: one test's event log from a FIFO simulation (produced by a passive
monitor: pin/flag change events with cycle timestamps, plus [TST] status
lines), together with the rule-book sections retrieved once for the feature
this test covers. You have no tools: judge only from what is given.

METHOD:
1. Read the log. Reconstruct what the stimulus did (accepted writes, accepted
   reads, blocked attempts, flag transitions) and the running occupancy.
2. Derive what the provided rules require given the observed stimulus, then
   judge the observed behavior against each applicable rule.

HARD RULES:
- Judge ONLY from the provided rule text and the log. No outside knowledge
  of FIFOs. If the rules and the log disagree with your intuition, they win.
- Events are logged at the first cycle the new value is observable (one
  cycle after the causing clock edge). Do not flag this sampling skew.
- The test FAILS if any applicable provided rule is violated.
- A log showing status=timeout, or lacking a status=completed line, is a
  FAILED test: verdict FAIL and the reason must start with TIMEOUT.
- If no provided rule covers an observed behavior, do not guess and do not
  invent rules: judge the rules you have and note the gap in the reason.
- Every citation must name a rule_id from the provided rules and quote the
  log evidence you used. Do not invent rules, values, or log lines.

Return CheckVerdict: verdict PASS or FAIL; when FAIL, reason must state the
violated rule and the exact log evidence.
"""

checker_agent = create_agent(
    model=checker_llm,
    tools=[],   # rules arrive with the log — retrieved once, from the feature
    system_prompt=CHECKER_PROMPT,
    middleware=[
        ModelCallLimitMiddleware(run_limit=CHECKER_MODEL_CALL_LIMIT,
                                 exit_behavior="end"),
    ],
    response_format=CheckVerdict,
    name="rule_checker",
)


def sanitize_verdict(verdict: CheckVerdict,
                     retrieved_ids: list[str]) -> tuple[CheckVerdict, list[str]]:
    """Citation-provenance guard: cited rules must have been retrieved."""
    errors: list[str] = []
    kept = [c for c in verdict.citations if c.rule_id in retrieved_ids]
    dropped = [c.rule_id for c in verdict.citations if c.rule_id not in retrieved_ids]
    if dropped:
        errors.append(f"Citations dropped (never retrieved): {dropped}")
    if verdict.verdict == "FAIL" and not verdict.reason:
        errors.append("FAIL verdict without a reason.")
    return verdict.model_copy(update={"citations": kept}), errors


def test_feature(test_name: str) -> str:
    """The feature this test covers, captured when testgen retrieved the test.
    Falls back to the test name itself at the gates (no testgen ran)."""
    return TESTGEN_RECORDS.get(test_name, {}).get("feature") or test_name


async def run_checker(test_name: str) -> CheckRecord:
    test_name = sanitize_name(test_name)
    log_path = LOG_DIR / f"{test_name}.log"
    if not log_path.exists():
        return CheckRecord(
            verdict=CheckVerdict(test_name=test_name, verdict="FAIL",
                                 reason="No log file was produced for this test."),
            errors=["missing log file"],
        )
    log_text = log_path.read_text()
    # ONE retrieval per test: the feature under test is the query
    feature = test_feature(test_name)
    rules = hybrid_rule_search(feature)
    rules_block = ("\n\nRule-book sections retrieved for the feature under test:\n"
                   + json.dumps(rules, indent=1))
    try:
        result = await checker_agent.ainvoke(
            {"messages": [{"role": "user", "content": (
                f"Test: {test_name}\nFeature under test: {feature}"
                f"{rules_block}\n\nEvent log:\n{log_text}"
            )}]}
        )
        verdict = CheckVerdict.model_validate(result["structured_response"])
        verdict = verdict.model_copy(update={"test_name": test_name})
        rule_ids = [r.get("rule_id", "?") for r in rules]
        chunks = [f"[{r.get('rule_id', '?')}] {r.get('text', '')}" for r in rules]
        verdict, errors = sanitize_verdict(verdict, rule_ids)
        return CheckRecord(verdict=verdict, retrieval_query=feature,
                           retrieved_rule_ids=rule_ids,
                           retrieved_chunks=chunks, test_log=log_text,
                           errors=errors)
    except Exception as exc:
        return CheckRecord(
            verdict=CheckVerdict(test_name=test_name, verdict="FAIL",
                                 reason=f"Checker did not complete: {exc}"),
            test_log=log_text,
            errors=[f"{type(exc).__name__}: {exc}"],
        )

print("Checker agent + provenance sanitizer ready.")


# side channels: full CheckRecords are too large to route through the router
# LLM's context; the compact verdict JSON goes to the router, the full record
# (log, chunks, tuple for RAGAS) is stored here, keyed by test_name.
CHECK_RECORDS: dict[str, CheckRecord] = {}


@tool(
    "run_check_agent",
    description=(
        "Delegate ONE completed test to a CheckerAgent, which retrieves the "
        "applicable rules and judges the test's event log. Input: the full "
        "test_name (e.g. T3_fill_sixteen). Returns the compact verdict JSON."
    ),
)
async def run_check_agent(test_name: str) -> str:
    record = await run_checker(test_name.strip())
    CHECK_RECORDS[record.verdict.test_name] = record
    return record.verdict.model_dump_json(indent=1)


TESTCHECK_ROUTER_PROMPT = f"""
You are TestCheckRouterLLM in an ASIC verification pipeline. Today is {TODAY}.

You receive the list of test_names whose simulations completed. Every one of
them must be checked exactly once. Work in waves: each model turn, emit
min(remaining, {VER_PARALLEL_LIMIT}) run_check_agent tool calls TOGETHER so
they execute in parallel, one test per call, and keep invoking further waves
until no tests remain.

Each call returns that checker's verdict. Do not re-judge or alter verdicts:
the checker is the authority on pass/fail. Your job is dispatch and
collection. Return CheckPhaseReport: checked = test_names whose check
completed (returned any verdict); failed_to_check = test_names whose check
call errored and returned no verdict.
"""

testcheck_router = create_agent(
    model=llm,
    tools=[run_check_agent],
    system_prompt=TESTCHECK_ROUTER_PROMPT,
    middleware=[
        ModelCallLimitMiddleware(run_limit=ROUTER_MODEL_CALL_LIMIT,
                                 exit_behavior="end"),
        ToolCallLimitMiddleware(tool_name="run_check_agent", run_limit=12,
                                exit_behavior="continue"),
    ],
    response_format=CheckPhaseReport,
    name="testcheck_router",
)
print("TestCheckRouterLLM ready.")


### Gate 5 — one CheckerAgent verdict (first LLM spend)

Judges the gate-4 log against retrieved rules. Inspect: verdict, reason grounded
in the log, citations only from retrieved rule ids. The record is registered so
gate 6 can score it.


In [23]:
# GATE 5: one checker call — checker LLM + rule retrieval
name = globals().get("GATE_TEST") or input("test name to check (blank=skip): ").strip()
if name:
    GATE_TEST = name
    gate_record = await run_checker(name)
    CHECK_RECORDS[gate_record.verdict.test_name] = gate_record
    print(gate_record.verdict.model_dump_json(indent=1))
    print("retrieved rule ids:", gate_record.retrieved_rule_ids)


## RAGAS — the Trust Gate Before Reporting

One RAGAS node, three scorings, one judge (`llm_factory` + `RunConfig`, per the
absorbed course idiom): **testgen retrieval** (feature query vs retrieved test-plan
chunk, from the `TESTGEN_RECORDS` side channel), **rules retrieval** (checking query
vs retrieved rule chunks), and **verdicts** (faithfulness of the checker's verdict
against rules + log). Retrieval scorings use context precision/recall; verdicts use
faithfulness/answer relevancy. `answer_correctness` stays inactive — no reference
verdicts can exist before a test runs.


In [ ]:
# --- RAGAS scoring (per-row .ascore idiom; sync judge with async bridge) ----
# ragas 0.4.x still imports a Vertex-AI shim that langchain-community 1.x no
# longer ships; stub that one module so the import succeeds.
import sys as _sys, types as _types
if "langchain_community.chat_models.vertexai" not in _sys.modules:
    try:
        import langchain_community.chat_models.vertexai  # noqa: F401
    except ModuleNotFoundError:
        _stub = _types.ModuleType("langchain_community.chat_models.vertexai")
        _stub.ChatVertexAI = type("ChatVertexAI", (), {})
        _sys.modules["langchain_community.chat_models.vertexai"] = _stub

import asyncio
from concurrent.futures import ThreadPoolExecutor

import instructor
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    Faithfulness,
)

# Human-authored answer key: which rules apply to each test (from the rule book).
APPLICABLE_RULES: dict[str, list[str]] = {
    "T1": ["R1"],
    "T2": ["R2", "R6", "R7"],
    "T3": ["R3", "R4"],
    "T4": ["R8", "R4", "R2"],
    "T5": ["R2", "R5", "R6", "R7"],
    "T6": ["R9", "R5"],
    "T7": ["R2", "R3"],
    "T8": ["R10", "R2", "R3"],
    "T9": ["R2", "R5", "R6"],
}


def build_sync_judge_llm():
    """Sync Gateway-free judge with an async-safe adapter (course session-6
    pattern): every real request is synchronous; only the Ragas coroutine
    boundary is bridged."""
    judge = llm_factory(
        JUDGE_MODEL_NAME,
        provider="openai",
        client=OpenAI(),
        mode=instructor.Mode.TOOLS,
    )
    # gpt-5.x rejects max_tokens; the current parameter is max_completion_tokens
    judge.model_args = {"max_completion_tokens": 4096, "max_retries": 3}

    async def agenerate_from_sync(prompt, response_model):
        return await asyncio.to_thread(
            judge.generate, prompt=prompt, response_model=response_model,
        )

    judge.agenerate = agenerate_from_sync
    return judge


def _run_coro_off_loop(async_function, *args):
    """Jupyter owns an event loop; run Ragas coroutines on a worker thread."""
    def invoke():
        return asyncio.run(async_function(*args))
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(invoke).result()


def _verdict_text(rec: CheckRecord) -> str:
    text = f"VERDICT: {rec.verdict.verdict}."
    if rec.verdict.reason:
        text += f" REASON: {rec.verdict.reason}"
    for c in rec.verdict.citations:
        text += f" [{c.rule_id}] {c.finding}"
    return text


def testgen_rows() -> list[dict]:
    """Scoring 1 — testgen retrieval: right test-plan chunk per feature query."""
    rows = []
    for test_name, rec in TESTGEN_RECORDS.items():
        chunk = rec["retrieved_contexts"][0] if rec["retrieved_contexts"] else ""
        fm = re.search(r"Feature under test:\s*(.+?)(?:Description:|$)", chunk, re.DOTALL)
        reference = f"The test plan covers this feature with {test_name}."
        if fm:
            reference += " Feature under test: " + " ".join(fm.group(1).split())
        rows.append({"test_id": test_name,
                     "user_input": rec["user_input"],
                     "retrieved_contexts": list(rec["retrieved_contexts"]),
                     "response": rec["response"],
                     "reference": reference})
    return rows


def rules_rows(records: list[CheckRecord]) -> list[dict]:
    """Scoring 2 — rules retrieval: right rules fetched for each judgment."""
    rows = []
    for rec in records:
        tname = rec.verdict.test_name
        rows.append({
            "test_id": tname,
            "user_input": rec.retrieval_query or f"rules applicable to test {tname}",
            "retrieved_contexts": rec.retrieved_chunks or ["(no retrieval)"],
            "response": "Retrieved rules: " + ", ".join(rec.retrieved_rule_ids or ["none"]),
            "reference": "Applicable rules: "
                         + ", ".join(APPLICABLE_RULES.get(tname.split("_")[0], [])),
        })
    return rows


def verdict_rows(records: list[CheckRecord]) -> list[dict]:
    """Scoring 3 — verdicts: grounded in the retrieved rules + the log."""
    rows = []
    for rec in records:
        tname = rec.verdict.test_name
        rows.append({
            "test_id": tname,
            "user_input": rec.retrieval_query or f"rules applicable to test {tname}",
            "retrieved_contexts": (rec.retrieved_chunks or ["(no retrieval)"])
                                  + [rec.test_log],
            "response": _verdict_text(rec),
            "reference": "Applicable rules: "
                         + ", ".join(APPLICABLE_RULES.get(tname.split("_")[0], [])),
        })
    return rows


async def _score_all(records: list[CheckRecord]) -> dict:
    judge = build_sync_judge_llm()
    from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
    ragas_embeddings = RagasOpenAIEmbeddings(client=OpenAI(), model="text-embedding-3-small")
    cp = ContextPrecision(llm=judge)
    cr = ContextRecall(llm=judge)
    fa = Faithfulness(llm=judge)
    ar = AnswerRelevancy(llm=judge, embeddings=ragas_embeddings)

    async def safe(coro):
        try:
            return round((await coro).value, 4)
        except Exception as exc:
            return f"error: {type(exc).__name__}"

    summary: dict = {}

    async def score_table(name, rows, scorers):
        if not rows:
            summary[name] = {"skipped": "no records"}
            return
        per_test = []
        for row in rows:
            entry = {"test_id": row["test_id"]}
            for metric_name, coro_fn in scorers:
                entry[metric_name] = await safe(coro_fn(row))
            per_test.append(entry)
        table = {"per_test": per_test}
        for metric_name, _ in scorers:
            vals = [e[metric_name] for e in per_test if isinstance(e[metric_name], float)]
            if vals:
                table[f"{metric_name}_mean"] = round(sum(vals) / len(vals), 4)
        summary[name] = table

    retrieval_scorers = [
        ("context_precision", lambda r: cp.ascore(
            user_input=r["user_input"], reference=r["reference"],
            retrieved_contexts=r["retrieved_contexts"])),
        ("context_recall", lambda r: cr.ascore(
            user_input=r["user_input"], retrieved_contexts=r["retrieved_contexts"],
            reference=r["reference"])),
    ]
    verdict_scorers = [
        ("faithfulness", lambda r: fa.ascore(
            user_input=r["user_input"], response=r["response"],
            retrieved_contexts=r["retrieved_contexts"])),
        ("answer_relevancy", lambda r: ar.ascore(
            user_input=r["user_input"], response=r["response"])),
    ]

    await score_table("testgen_retrieval", testgen_rows(), retrieval_scorers)
    await score_table("rules_retrieval", rules_rows(records), retrieval_scorers)
    await score_table("verdicts", verdict_rows(records), verdict_scorers)

    # reporter contract unchanged: verdict trust scores also at top level
    summary.update({k: v for k, v in summary.get("verdicts", {}).items()})
    return summary


def run_ragas(records: list[CheckRecord]) -> dict:
    """One RAGAS stage, three scorings. Returns a summary dict; never raises."""
    try:
        summary = _run_coro_off_loop(_score_all, records)
        for name in ("testgen_retrieval", "rules_retrieval", "verdicts"):
            print(f"--- {name} ---")
            print(json.dumps(summary.get(name, {}), indent=1))
        return summary
    except Exception as exc:
        return {"error": f"RAGAS did not complete: {type(exc).__name__}: {exc}"}

print("RAGAS stage ready (per-row ascore; sync judge; vertexai stub).")


RAGAS stage ready (per-row ascore; sync judge; vertexai stub).


### Gate 6 — RAGAS on the gate-5 record (judge LLM spend)

Scores whatever is in `CHECK_RECORDS` — after gate 5 that is exactly one record,
so you see the four metrics on a single verdict before the full run.


In [59]:
# GATE 6: RAGAS on accumulated records — judge LLM
if CHECK_RECORDS:
    print(run_ragas(list(CHECK_RECORDS.values())))
else:
    print("no CheckRecords yet — run gate 5 first")


/tmp/ipykernel_2894/409980888.py:132: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = embedding_factory(
API call failed on attempt 1: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}
Max retries exceeded. Total attempts: 1, Last error: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}
API call failed on attempt 1: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not suppo

--- testgen_retrieval ---
{
 "skipped": "no records"
}
--- rules_retrieval ---
{
 "per_test": [
  {
   "test_id": "T1_cold_start",
   "context_precision": "error: InstructorRetryException",
   "context_recall": "error: InstructorRetryException"
  }
 ]
}
--- verdicts ---
{
 "per_test": [
  {
   "test_id": "T1_cold_start",
   "faithfulness": "error: InstructorRetryException",
   "answer_relevancy": "error: InstructorRetryException"
  }
 ]
}
{'testgen_retrieval': {'skipped': 'no records'}, 'rules_retrieval': {'per_test': [{'test_id': 'T1_cold_start', 'context_precision': 'error: InstructorRetryException', 'context_recall': 'error: InstructorRetryException'}]}, 'verdicts': {'per_test': [{'test_id': 'T1_cold_start', 'faithfulness': 'error: InstructorRetryException', 'answer_relevancy': 'error: InstructorRetryException'}]}, 'per_test': [{'test_id': 'T1_cold_start', 'faithfulness': 'error: InstructorRetryException', 'answer_relevancy': 'error: InstructorRetryException'}]}


## Reporter Agent

Compiles the final report **after RAGAS**: verdicts with reasons for failures (pass
rows stay terse), generation/run failures called out separately from checker FAILs,
and per-test trust scores. A deterministic fallback report is produced if the LLM
stage fails — the pipeline never ends without a report.


In [26]:
REPORTER_PROMPT = f"""
You are the reporter of an ASIC verification pipeline. Today is {TODAY}.

You receive: check verdicts (with reasons and rule citations), generation
failures, run failures, and RAGAS trust scores per verdict.

Produce FinalReport:
- results_table_markdown: one row per requested test with columns
  Test | Outcome | Reason / notes | Faithfulness | Context recall.
  Outcome is PASS, FAIL, GEN-FAIL, or RUN-FAIL. PASS and FAIL come verbatim
  from the checker verdicts — you never judge a test yourself (a timed-out
  test arrives already judged: the checker's FAIL with a TIMEOUT reason).
  GEN-FAIL / RUN-FAIL are only for tests that never produced a log.
  PASS rows need no prose.
- failures_detail_markdown: for each FAIL, the violated rule, the log
  evidence, and the checker's reasoning; for GEN/RUN failures, the log tail.
- trust_notes: flag any verdict whose faithfulness or recall is low; the
  reader should know which verdicts to double-check by hand.
- Do not invent tests, rules, scores, or log content.
"""

reporter_agent = create_agent(
    model=writer_llm,
    tools=[],
    system_prompt=REPORTER_PROMPT,
    response_format=FinalReport,
    name="reporter",
)


def fallback_report(state: "PipelineState") -> FinalReport:
    lines = ["| Test | Outcome | Notes |", "|---|---|---|"]
    for rec in state.get("check_records", []):
        v = rec.verdict
        lines.append(f"| {v.test_name} | {v.verdict} | {v.reason or ''} |")
    for g in state.get("gen_phase", GenPhaseReport()).failed:
        lines.append(f"| {g.test_name or g.feature} | GEN-FAIL | {g.error or g.gen_log_tail[:120]} |")
    for r in state.get("run_phase", RunPhaseReport()).failed:
        lines.append(f"| {r.test_name} | RUN-FAIL | {r.error or r.status} |")
    return FinalReport(
        title="FIFO regression report (deterministic fallback)",
        summary="Reporter LLM unavailable; table assembled by code.",
        results_table_markdown="\n".join(lines),
    )


## The LangGraph Pipeline

LangGraph owns the lifecycle; agents own judgment. Checking fans out with bounded
concurrency (`VER_PARALLEL_LIMIT` checkers at a time). A build/ingest failure routes
directly to the reporter with the log — the stop-and-tell-the-user path.


In [ ]:
async def orchestrate_node(state: PipelineState, config: RunnableConfig) -> dict:
    result = await orchestrator_agent.ainvoke(
        {"messages": [{"role": "user", "content":
            "Prepare the pipeline: build the model and ingest the documents, "
            "in parallel, then judge whether to proceed."}]},
        config=config,
    )
    decision = OrchestratorDecision.model_validate(result["structured_response"])

    return {"orchestrator": decision}


def route_after_orchestrate(state: PipelineState) -> Literal["generate", "report"]:
    return "generate" if state["orchestrator"].proceed else "report"


async def generate_node(state: PipelineState, config: RunnableConfig) -> dict:
    request = (state.get("user_query") or "").strip() or "all features"
    try:
        result = await testgen_router.ainvoke(
            {"messages": [{"role": "user", "content":
                f"Features the user wants tested:\n{request}"}]},
            config=config,
        )
        report = GenPhaseReport.model_validate(result["structured_response"])
        return {"gen_phase": report}
    except Exception as exc:
        return {"gen_phase": GenPhaseReport(
                    failed=[GenResult(feature=request, status="fail",
                                      error=f"router: {exc}")]),
                "errors": state.get("errors", []) + [f"gen: {exc}"]}


async def run_node(state: PipelineState, config: RunnableConfig) -> dict:
    todo = state["gen_phase"].passed
    if not todo:
        return {"run_phase": RunPhaseReport()}
    try:
        result = await testrun_router.ainvoke(
            {"messages": [{"role": "user", "content":
                f"Run these tests:\n{json.dumps(todo, indent=1)}"}]},
            config=config,
        )
        report = RunPhaseReport.model_validate(result["structured_response"])
        return {"run_phase": report}
    except Exception as exc:
        return {"run_phase": RunPhaseReport(
                    failed=[RunResult(test_name=t, status="not_run",
                                      error=f"router: {exc}") for t in todo]),
                "errors": state.get("errors", []) + [f"run: {exc}"]}


async def check_node(state: PipelineState, config: RunnableConfig) -> dict:
    done = state["run_phase"].done
    if not done:
        return {"check_records": []}
    CHECK_RECORDS.clear()
    errors = list(state.get("errors", []))
    try:
        result = await testcheck_router.ainvoke(
            {"messages": [{"role": "user", "content":
                "Check these completed tests:\n" + json.dumps(done, indent=1)}]},
            config=config,
        )
        phase = CheckPhaseReport.model_validate(result["structured_response"])
        missing = [t for t in done if t not in CHECK_RECORDS]
        if missing:
            errors.append(f"router never dispatched checks for: {missing}")
    except Exception as exc:
        errors.append(f"check router: {type(exc).__name__}: {exc}")
        # fallback: check directly so verdicts are never lost
        for t in done:
            if t not in CHECK_RECORDS:
                CHECK_RECORDS[t] = await run_checker(t)
    records = [CHECK_RECORDS[t] for t in done if t in CHECK_RECORDS]
    return {"check_records": records, "errors": errors}


def ragas_node(state: PipelineState) -> dict:
    return {"ragas_summary": run_ragas(state.get("check_records", []))}


async def report_node(state: PipelineState, config: RunnableConfig) -> dict:
    if not state["orchestrator"].proceed:
        # stop-and-surface path: user must see why nothing ran
        report = FinalReport(
            title="Pipeline aborted before testing",
            summary=state["orchestrator"].reason,
            results_table_markdown="(no tests were generated or run)",
            failures_detail_markdown="```\n"
                + state["orchestrator"].build_log_tail + "\n```",
        )
        return {"report": report}
    payload = {
        "verdicts": [r.verdict.model_dump() for r in state.get("check_records", [])],
        "gen_failures": state["gen_phase"].model_dump()["failed"],
        "run_failures": state["run_phase"].model_dump()["failed"],
        "ragas": state.get("ragas_summary", {}),
    }
    try:
        result = await reporter_agent.ainvoke(
            {"messages": [{"role": "user",
                           "content": json.dumps(payload, indent=1)}]},
            config=config,
        )
        return {"report": FinalReport.model_validate(result["structured_response"])}
    except Exception as exc:
        return {"report": fallback_report(state),
                "errors": state.get("errors", []) + [f"report: {exc}"]}


builder = StateGraph(PipelineState)
builder.add_node("orchestrate", orchestrate_node)
builder.add_node("generate", generate_node)
builder.add_node("run", run_node)
builder.add_node("check", check_node)
builder.add_node("ragas", ragas_node)
builder.add_node("report", report_node)

builder.add_edge(START, "orchestrate")
builder.add_conditional_edges("orchestrate", route_after_orchestrate,
                              {"generate": "generate", "report": "report"})
builder.add_edge("generate", "run")
builder.add_edge("run", "check")
builder.add_edge("check", "ragas")
builder.add_edge("ragas", "report")
builder.add_edge("report", END)

verification_graph = builder.compile(checkpointer=InMemorySaver())
print("Compiled verification_graph.")


## Synthetic Tests 
 the citation-provenance sanitizer, the
test-plan/rule-book splitters, and the run-status classifier tested here


In [69]:
# citation provenance: cited rules must have been retrieved
v = CheckVerdict(test_name="T3_fill_sixteen", verdict="FAIL", reason="full asserted early",
                 citations=[RuleCitation(rule_id="R4", finding="ok"),
                            RuleCitation(rule_id="R99", finding="invented")])
sv, errs = sanitize_verdict(v, retrieved_ids=["R4", "R3"])
assert [c.rule_id for c in sv.citations] == ["R4"]
assert errs and "R99" in errs[0]

# name sanitizer + bare-name resolution against the tb
assert sanitize_name("T7 — lap_the_buffer") == "T7_lap_the_buffer"
assert sanitize_name("full_drain") == "T5_full_drain"

# docling chunk conversion + section grouping (synthetic chunks, no PDF)
class _Meta:
    def __init__(self, headings): self.headings = headings
class _Chunk:
    def __init__(self, headings, text): self.meta, self.text = _Meta(headings), text

raw = [_Chunk(["4. Test cases", "4.1 » T1 — cold_start"], "Stimulus: drive rst low."),
       _Chunk(["4. Test cases", "4.1 » T1 — cold_start"], "Pass criteria: empty=1."),
       _Chunk(["4. Test cases", "4.2 » T2 — one_in_one_out"], "Stimulus: one write.")]
chunks = [convert_chunk(c) for c in raw]
assert chunks[0]["chunk_text"].startswith("4. Test cases > 4.1 » T1 — cold_start")
grouped = group_sections(chunks, TEST_HEADING)
assert list(grouped) == ["T1", "T2"] and len(grouped["T1"]["parts"]) == 2
rule = group_sections([convert_chunk(_Chunk(["3. Data rules", "R2 — FIFO order"],
                                            "Words emerge in write order."))], RULE_HEADING)
assert list(rule) == ["R2"]

# BM25Okapi + verbatim RRF on a toy corpus
toy = [Document(page_content=t, metadata={"page_id": p, "rule_id": p,
                                          "chunk_id": f"c{n}"})
       for n, (p, t) in enumerate([
           ("R6", "rd_data valid one cycle after accepted read rd_en"),
           ("R4", "full occupancy 16 blocks writes wr_en"),
           ("R1", "reset empty rd_data zero")])]
toy_bm25 = BM25Okapi([tokenize(d.page_content) for d in toy])
scores = toy_bm25.get_scores(tokenize("Accepted READ rd_en latency"))
ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
lex = [as_retrieved_document(toy[i], float(scores[i])) for i in ranked]
fused = reciprocal_rank_fusion([lex, list(reversed(lex))], limit=2)
assert lex[0].metadata["rule_id"] == "R6"          # case-insensitive tokenize
assert "rrf_score" in fused[0].metadata

# ragas row composition: rules and log stay separable, log rides last
rec = CheckRecord(
    verdict=CheckVerdict(test_name="T1_cold_start", verdict="PASS"),
    retrieval_query="reset rules",
    retrieved_rule_ids=["R1"],
    retrieved_chunks=["[R1] after reset empty=1"],
    test_log="[EVT] cycle=2 pin=rst_n 0 -> 1",
)
row = verdict_rows([rec])[0]
assert row["retrieved_contexts"][0] == "[R1] after reset empty=1"
assert row["retrieved_contexts"][-1] == rec.test_log

print("Synthetic tests passed.")


Synthetic tests passed.


## Full Pipeline Run

The USER names the features to test — in a sentence, at the prompt below. The
orchestrator preps and judges readiness, the routers dispatch waves of agents,
the checkers rule, RAGAS scores, the reporter writes. Nothing test-specific
lives in this cell.

In [70]:
# FULL RUN — the user names the features; the pipeline does the rest.
# e.g. "Test reset."  |  "Test FIFO ordering, wraparound and underflow."
user_query = input("Features to test: ").strip()
if user_query:
    final_state = await verification_graph.ainvoke(
        {"user_query": user_query},
        config={"configurable": {"thread_id": "fifo-regression"}},
    )
    print("orchestrator:", final_state["orchestrator"].model_dump_json(indent=1))
    gp = final_state.get("gen_phase")
    rp = final_state.get("run_phase")
    print("gen passed:", gp.passed if gp else "—")
    print("run done:", rp.done if rp else "—")
    print("verdicts:", [f"{r.verdict.test_name}:{r.verdict.verdict}"
                        for r in final_state.get("check_records", [])])


[INFO] 2026-07-16 15:31:11,271 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-16 15:31:11,274 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-16 15:31:11,317 [RapidOCR] download_file.py:60: File exists and is valid: /home/sajalosta/.venvs/fifo/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-16 15:31:11,318 [RapidOCR] main.py:50: Using /home/sajalosta/.venvs/fifo/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-16 15:31:11,573 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-16 15:31:11,574 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-16 15:31:11,577 [RapidOCR] download_file.py:60: File exists and is valid: /home/sajalosta/.venvs/fifo/lib/python3.13/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-16 15:31:11,578 [RapidOCR] main.py:50: Using /home/sajalosta/.venvs/fifo/lib/python3.13/site-packages/rapidocr/models/ch

--- testgen_retrieval ---
{
 "per_test": [
  {
   "test_id": "T6_read_on_dry",
   "context_precision": 1.0,
   "context_recall": 1.0
  },
  {
   "test_id": "T1_cold_start",
   "context_precision": 1.0,
   "context_recall": 1.0
  },
  {
   "test_id": "T3_fill_sixteen",
   "context_precision": 1.0,
   "context_recall": 1.0
  },
  {
   "test_id": "T7_lap_the_buffer",
   "context_precision": 1.0,
   "context_recall": 1.0
  },
  {
   "test_id": "T4_seventeenth_write",
   "context_precision": 1.0,
   "context_recall": 1.0
  }
 ],
 "context_precision_mean": 1.0,
 "context_recall_mean": 1.0
}
--- rules_retrieval ---
{
 "per_test": [
  {
   "test_id": "T1_cold_start",
   "context_precision": 1.0,
   "context_recall": 1.0
  },
  {
   "test_id": "T6_read_on_dry",
   "context_precision": 1.0,
   "context_recall": 1.0
  },
  {
   "test_id": "T3_fill_sixteen",
   "context_precision": 1.0,
   "context_recall": 0.0
  },
  {
   "test_id": "T7_lap_the_buffer",
   "context_precision": 1.0,
   "context_re

In [71]:
# The FinalReport, rendered
rep = final_state.get("report") if "final_state" in globals() else None
if rep:
    display(Markdown("\n\n".join(filter(None, [
        f"# {getattr(rep, 'title', 'FIFO Regression Report')}",
        rep.summary,
        rep.results_table_markdown,
        rep.failures_detail_markdown,
        f"**Trust notes:** {rep.trust_notes}" if rep.trust_notes else "",
    ]))))
else:
    print("No report in state — run the pipeline cell first.")


# ASIC Verification Pipeline Final Report

All requested tests produced checker verdicts and completed logs. Four tests passed. No generation or run failures were reported. RAGAS indicates one verdict with slightly lower faithfulness than the rest, and several verdicts with retrieval recall below 1.0 should be double-checked by hand.

| Test | Outcome | Reason / notes | Faithfulness | Context recall |
|---|---|---|---:|---:|
| T1_cold_start | PASS |  | 0.9375 | 1.0 |
| T6_read_on_dry | PASS |  | error: IncompleteOutputException | 1.0 |
| T3_fill_sixteen | PASS |  | error: IncompleteOutputException | 0.0 |
| T7_lap_the_buffer | PASS |  | error: IncompleteOutputException | 0.0 |
| T4_seventeenth_write | PASS |  | error: IncompleteOutputException | 0.3333 |

No FAIL verdicts were provided.

No GEN-FAIL or RUN-FAIL entries were provided.

**Trust notes:** Double-check these verdicts by hand due to weaker trust signals: T1_cold_start has faithfulness 0.9375, and T3_fill_sixteen, T7_lap_the_buffer, and T4_seventeenth_write have rules-retrieval context recall below 1.0 (0.0, 0.0, and 0.3333 respectively). T6_read_on_dry also has faithfulness reported as an error rather than a numeric score, so its output should be reviewed manually. No verdicts were reported as low on the available numeric faithfulness threshold except T1_cold_start being the only scored one.

---
Deferred work lives in `FUTURE_FEATURES.md`. The two PDFs in `docs/` are the
knowledge; `hw/` is the design under test; everything else is regenerated per run.